# Level 3 — Advanced: Visualisation and End-to-End Analysis

**Course:** Python & Data Analytics &nbsp;|&nbsp; **Program:** B.Tech Technical Training – CSE A & B

**What this set covers**
- **Part A – Visualisation toolkit:** line, bar, histogram, box, scatter, heatmap and multi-panel figures with Matplotlib and Seaborn
- **Part B – End-to-end case study:** a complete question → clean → analyse → visualise → conclude workflow
- **Part C – Insight reporting:** turning numbers into clear statements a manager can act on

The setup cell gives you `load_sales()` and `load_students()`. They return **cleaned** data using the same steps you
wrote in Level 2, so you can focus on analysis and charts here.

**Rules for every chart in this set:** it must have a title, labelled axes (with units) and a legend
whenever there is more than one series. A chart without these will not get full marks in the projects.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = "../../datasets/"
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 4.5)
pd.set_option("display.float_format", "{:,.2f}".format)

SUBJECTS = ["Mathematics", "Physics", "Programming", "English", "Electronics"]


def load_sales():
    """Load sales_data.csv and apply the Level 2 cleaning steps."""
    df = pd.read_csv(DATA_DIR + "sales_data.csv").drop_duplicates().reset_index(drop=True)
    df["Region"] = df["Region"].str.strip().str.title()
    df["Unit_Price"] = pd.to_numeric(
        df["Unit_Price"].astype(str).str.replace("Rs.", "", regex=False), errors="coerce")
    for col in ["Unit_Price", "Quantity"]:
        df[col] = df[col].fillna(df.groupby("Product")[col].transform("median"))
    df["Quantity"] = df["Quantity"].round().astype(int)
    df["Customer_Segment"] = df["Customer_Segment"].fillna("Unknown")
    df["Order_Date"] = pd.to_datetime(df["Order_Date"])
    df["Month"] = df["Order_Date"].dt.month
    df["Revenue"] = (df["Quantity"] * df["Unit_Price"] * (1 - df["Discount"])).round(2)
    return df


def load_students():
    """Load student_performance.csv and apply the Level 2 cleaning steps."""
    df = pd.read_csv(DATA_DIR + "student_performance.csv").drop_duplicates().reset_index(drop=True)
    df["Gender"] = df["Gender"].str.strip().str.lower().map(
        {"m": "Male", "male": "Male", "f": "Female", "female": "Female"})
    for col in ["Attendance_Pct"] + SUBJECTS:
        df[col] = df[col].where(df[col].between(0, 100))
    return df


sales = load_sales()
students = load_students()
print("sales:", sales.shape, "| students:", students.shape)

## Part A — Visualisation toolkit

### Exercise 3.1 — Line chart: a trend over time

Plot **weekly** total revenue across 2024 as a line chart. Add a dashed horizontal line at the average weekly revenue. Show the y-axis in lakhs.

<details><summary>Hint</summary>

`sales.set_index('Order_Date').resample('W')['Revenue'].sum()` gives weekly totals. `ax.axhline(...)` draws the reference line.

</details>

In [ ]:
# Your code here

### Exercise 3.2 — Bar chart: comparing categories

Make a **horizontal** bar chart of total revenue by `Payment_Mode`, sorted so the largest bar is at the top. Write the value (in lakhs) at the end of each bar.

<details><summary>Hint</summary>

Horizontal bars are easier to read when the category names are long. `ax.bar_label(ax.containers[0], fmt=...)` adds the values.

</details>

In [ ]:
# Your code here

### Exercise 3.3 — Histogram: the shape of a distribution

Plot the distribution of order `Revenue`. It is heavily skewed, so plot it a second time on a **log scale**. In a markdown cell, describe what each version shows.

<details><summary>Hint</summary>

`sns.histplot(data=sales, x='Revenue', log_scale=True, kde=True)`

</details>

In [ ]:
# Your code here

### Exercise 3.4 — Box plot: comparing distributions

Compare the Programming marks of **hostellers vs day scholars**, split by `Section`, using a box plot. Which group has the highest median? Which has the widest spread?

<details><summary>Hint</summary>

`sns.boxplot(data=students, x='Accommodation', y='Programming', hue='Section')`

</details>

In [ ]:
# Your code here

### Exercise 3.5 — Scatter plot: relationships

Plot `Attendance_Pct` against `Programming` marks, coloured by `Section`, with a regression line. Calculate the correlation coefficient and say in words how strong the relationship is.

<details><summary>Hint</summary>

`sns.regplot` draws the line; `sns.scatterplot` with `hue` gives the colours. `students['Attendance_Pct'].corr(students['Programming'])`

</details>

In [ ]:
# Your code here

### Exercise 3.6 — Heatmap: many relationships at once

Draw a correlation heatmap of attendance, study hours, previous GPA and the five subjects. Show the numbers in the cells and use a diverging colour map centred on 0. Which factor is most strongly linked to marks?

<details><summary>Hint</summary>

`sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1)`

</details>

In [ ]:
# Your code here

### Exercise 3.7 — Multi-panel figure: a mini dashboard

Load `weather_data.csv`. For **Hyderabad in 2024**, build a 2×2 figure showing:
1. daily max and min temperature (line)
2. monthly total rainfall (bar)
3. distribution of humidity (histogram)
4. max temperature vs humidity (scatter)

Give the whole figure a main title with `fig.suptitle`. Remember to remove the `-99` error codes and the duplicates first.

<details><summary>Hint</summary>

`fig, axes = plt.subplots(2, 2, figsize=(13, 8))`, then draw on `axes[0, 0]`, `axes[0, 1]` and so on.

</details>

In [ ]:
# Your code here

### Exercise 3.8 — Saving figures

Re-create the bar chart from 3.2 and save it as `revenue_by_payment_mode.png` at 150 dpi. Then save it again as a `.svg` file. When is each format better?

<details><summary>Hint</summary>

Call `fig.savefig(...)` **before** `plt.show()`, and use `bbox_inches='tight'` so labels are not cut off.

</details>

In [ ]:
# Your code here

## Part B — End-to-end case study: *Who buys, and how do they pay?*

The marketing team asks: **"Which customer segments bring in the most revenue, how do they prefer to pay, and do
discounts make people buy more?"** Work through the five steps below. Each step has its own exercise.

### Exercise 3.9 — Step 1 — Frame the questions

In a markdown cell, rewrite the marketing team's request as **three precise questions** that data can answer. For each one, name the columns and the calculation you will need.

In [ ]:
# Write your three questions in a markdown cell above or below this one.

### Exercise 3.10 — Step 2 — Check the data you need

Check the columns needed for your questions: missing values, the distinct values in each category, and how many orders have the segment `'Unknown'`. Decide whether to keep or exclude the `'Unknown'` orders, and justify your decision.

In [ ]:
# Your code here

### Exercise 3.11 — Step 3 — Analyse (Q1 and Q2)

Answer Q1 with a summary table per segment: orders, revenue (lakh), share of revenue (%) and average order value. Answer Q2 with a table showing what percentage of each segment's revenue comes from each payment mode (each row should add up to 100%).

<details><summary>Hint</summary>

For Q2, `pd.crosstab(index, columns, values, aggfunc='sum', normalize='index')`.

</details>

In [ ]:
# Your code here

### Exercise 3.12 — Step 3 — Analyse (Q3)

Compare the **average quantity per order at each discount level**, separately for each category. Why is it important to split by category here, rather than looking at all orders together?

In [ ]:
# Your code here

### Exercise 3.13 — Step 4 — Visualise the answers

Create **one figure with two panels** that a marketing manager could understand in 10 seconds:
- left: revenue by segment (bar)
- right: payment-mode mix for each segment (100% stacked bar)

<details><summary>Hint</summary>

`pay_mix.plot(kind='barh', stacked=True, ax=axes[1])` makes a stacked bar from the crosstab.

</details>

In [ ]:
# Your code here

### Exercise 3.14 — Step 5 — Conclude

Write **three insights** in the format below, each backed by a number from your analysis, followed by **one recommendation**.

> **Insight:** what you found (with a number) → **So what:** why it matters to the business

In [ ]:
# Write your insights in a markdown cell. You may compute any supporting numbers here.

## Part C — Insight reporting

### Exercise 3.15 — Critique a chart

Run the cell below. It produces a chart that breaks several good-practice rules. List **at least four** problems, then redraw it properly.

In [ ]:
bad = sales.groupby("Region")["Revenue"].sum()
plt.figure(figsize=(5, 5))
plt.pie(bad, colors=["red", "red", "orange", "orange", "yellow"], startangle=90)
plt.show()

# Your improved chart here

---
## Before you move on
- [ ] Every chart has a title, axis labels with units, and a legend where needed
- [ ] Every insight in 3.14 is backed by a number
- [ ] You have said clearly what the data **cannot** prove (correlation vs causation)

You are now ready for the **mini-projects** in `../../mini-projects/`.